# Entrenamiento YOLO en Colab (YOLO11n · YOLO11s · YOLO26n)

Entrena tres variantes sobre el dataset de tarjetas.

- `YOLO11n` y `YOLO11s` → dataset **Cards**
- `YOLO26n` → dataset **Cards-2**
- Split reproducible `train / val / test` por separado
- Análisis de distribución de clases antes de entrenar
- Preview visual de augmentaciones antes de aplicarlas
- Export `.pt` y `.onnx` con guardado automático en Drive

## 1) Instalación

In [ ]:
%pip install -q ultralytics albumentations opencv-python pyyaml onnx onnxsim roboflow

import os, shutil, random, yaml, glob, json, math, colorsys
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
from google.colab import drive

drive.mount('/content/drive')
print('Ultralytics listo')

## 1b) Descargar dataset desde Roboflow

Descarga el dataset directamente desde Roboflow para garantizar compatibilidad con los paquetes instalados en este entorno.

> Configura el secreto `ROBOFLOW_API_KEY` en **Colab → Secrets** antes de ejecutar.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

project = rf.workspace("thesis-s4jik").project("cards-4ceff")
version = project.version(3)
dataset = version.download("yolo26")  # se guarda en Cards-3

DATASET_YOLO11 = dataset.location
DATASET_YOLO26 = dataset.location

print("Dataset descargado con éxito.")
print("DATASET_YOLO11:", DATASET_YOLO11)
print("DATASET_YOLO26:", DATASET_YOLO26)

## 2) Configuración

In [ ]:
import torch

# ── Rutas ──────────────────────────────────────────────────────────────────
# DATASET_YOLO11 y DATASET_YOLO26 se definen en la sección 1b (descarga de Roboflow)
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Cards_Training_Output'

BASE_DIR         = Path('/content/cards_yolo')
YOLO11_SPLIT_DIR = BASE_DIR / 'yolo11_split_dataset'
YOLO26_SPLIT_DIR = BASE_DIR / 'yolo26_split_dataset'
YOLO11_AUG_DIR   = BASE_DIR / 'yolo11_split_dataset_aug'
YOLO26_AUG_DIR   = BASE_DIR / 'yolo26_split_dataset_aug'
RUNS_DIR         = BASE_DIR / 'runs'

# ── Dispositivo ────────────────────────────────────────────────────────────
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('⚠  No se detectó GPU. El entrenamiento en CPU será muy lento.')
    print('   Activa la GPU en Colab: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU')
else:
    print('GPU detectada: {}'.format(torch.cuda.get_device_name(0)))

# ── Split ──────────────────────────────────────────────────────────────────
VAL_RATIO  = 0.15
TEST_RATIO = 0.10
SEED       = 42

# ── Reproducibilidad ───────────────────────────────────────────────────────
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Entrenamiento ──────────────────────────────────────────────────────────
IMG_SIZE  = 640
EPOCHS    = 120
BATCH     = 16
PATIENCE  = 25
WORKERS   = 2

# ── Hiperparámetros adicionales (Ultralytics best practices) ───────────────
COSINE_LR = True   # cosine LR decay en lugar de lineal
NBS       = 64     # nominal batch size -> acumulacion de gradiente efectiva
MIXUP     = 0.1    # mezcla de imagenes para generalizar
DEGREES   = 10.0
TRANSLATE = 0.1
SCALE     = 0.5
FLIPLR    = 0.5
FLIPUD    = 0.0
HSV_H     = 0.015
HSV_S     = 0.7
HSV_V     = 0.4

# ── Albumentations offline ─────────────────────────────────────────────────
USE_OFFLINE_ALBUMENTATIONS = True
AUG_MULTIPLIER = 1  # copias sinteticas por imagen de train

BASE_DIR.mkdir(parents=True, exist_ok=True)
print('Configuracion aplicada  |  device={}'.format(DEVICE))

## 3) Utilidades

In [ ]:
# === Labels YOLO ============================================================

def read_yolo_labels(label_path):
    bboxes, class_ids = [], []
    if not Path(label_path).exists():
        return bboxes, class_ids
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, x, y, w, h = parts
            class_ids.append(int(cls))
            bboxes.append([float(x), float(y), float(w), float(h)])
    return bboxes, class_ids

def write_yolo_labels(label_path, bboxes, class_ids):
    with open(label_path, 'w') as f:
        for cls, box in zip(class_ids, bboxes):
            x, y, w, h = box
            f.write('{} {:.6f} {:.6f} {:.6f} {:.6f}\n'.format(cls, x, y, w, h))

# === Estructura de directorios ==============================================

def ensure_split_structure(root):
    for s in ['train', 'val', 'test']:
        (root / s / 'images').mkdir(parents=True, exist_ok=True)
        (root / s / 'labels').mkdir(parents=True, exist_ok=True)

def load_names_from_yaml(dataset_yaml):
    with open(dataset_yaml) as f:
        return yaml.safe_load(f)['names']

# === Split ==================================================================

def split_single_dataset(dataset_path, output_dir, val_ratio=0.15, test_ratio=0.10, seed=42):
    ensure_split_structure(output_dir)
    pairs = []
    for img in sorted(glob.glob(str(dataset_path) + '/train/images/*')):
        lbl = str(dataset_path) + '/train/labels/' + Path(img).stem + '.txt'
        if Path(lbl).exists():
            pairs.append((img, lbl))
    rnd = random.Random(seed)
    rnd.shuffle(pairs)
    n       = len(pairs)
    n_test  = int(n * test_ratio)
    n_val   = int(n * val_ratio)
    n_train = n - n_val - n_test
    splits  = {
        'train': pairs[:n_train],
        'val':   pairs[n_train:n_train + n_val],
        'test':  pairs[n_train + n_val:]
    }
    ds_name = Path(dataset_path).name
    for sname, items in splits.items():
        for i, (img, lbl) in enumerate(items):
            ext  = Path(img).suffix.lower()
            stem = ds_name + '_' + Path(img).stem + '_' + str(i)
            shutil.copy2(img, output_dir / sname / 'images' / (stem + ext))
            shutil.copy2(lbl, output_dir / sname / 'labels' / (stem + '.txt'))
    return {k: len(v) for k, v in splits.items()}

# === Augmentacion offline ===================================================

def apply_offline_augment(split_dir, aug_dir, aug_multiplier, transform):
    """Copia split_dir en aug_dir y genera aug_multiplier copias augmentadas de cada imagen de train."""
    if aug_dir.exists():
        shutil.rmtree(aug_dir)
    shutil.copytree(split_dir, aug_dir)
    img_dir = aug_dir / 'train' / 'images'
    lbl_dir = aug_dir / 'train' / 'labels'
    created = 0
    for img_path in sorted(img_dir.glob('*')):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        bboxes, cls_ids = read_yolo_labels(lbl_path)
        if not bboxes:
            continue
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        for k in range(aug_multiplier):
            aug = transform(image=image, bboxes=bboxes, class_labels=cls_ids)
            out = cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR)
            cv2.imwrite(str(img_dir / (img_path.stem + '_aug' + str(k) + '.jpg')), out)
            write_yolo_labels(
                lbl_dir / (img_path.stem + '_aug' + str(k) + '.txt'),
                aug['bboxes'], [int(c) for c in aug['class_labels']]
            )
            created += 1
    return created

def build_yaml(data_root, names, yaml_path):
    cfg = {
        'path':  str(data_root),
        'train': 'train/images',
        'val':   'val/images',
        'test':  'test/images',
        'nc':    len(names),
        'names': names
    }
    with open(yaml_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

# === Colores ================================================================

def _class_colors(n):
    return [
        tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / max(n, 1), 0.75, 0.9))
        for i in range(n)
    ]

# === Distribucion de clases =================================================

def show_class_distribution(split_dir, names, title=''):
    counts = {name: 0 for name in names}
    for lbl in sorted((split_dir / 'train' / 'labels').glob('*.txt')):
        _, cls_ids = read_yolo_labels(lbl)
        for c in cls_ids:
            if 0 <= c < len(names):
                counts[names[c]] += 1
    total = sum(counts.values())
    print('  Total instancias en train: {}'.format(total))
    keys   = list(counts.keys())
    vals   = list(counts.values())
    colors = ['#d62728' if v < 50 else '#1f77b4' for v in vals]
    fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.40)))
    bars = ax.barh(keys, vals, color=colors)
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_xlabel('Instancias')
    ttl = 'Distribucion de clases (train)  -  rojo = <50 instancias'
    ax.set_title((title + '  |  ' + ttl) if title else ttl)
    plt.tight_layout()
    plt.show()
    low = [k for k, v in counts.items() if v < 50]
    if low:
        print('  Clases con <50 instancias: {}'.format(low))
    return counts

# === Visualizar augmentaciones ==============================================

def visualize_augmentations(split_dir, transform, names, n=16):
    """Muestra n imagenes del train split con augmentaciones aplicadas y bboxes dibujados."""
    imgs = sorted((split_dir / 'train' / 'images').glob('*'))
    rng  = random.Random(SEED)
    rng.shuffle(imgs)
    imgs   = imgs[:n]
    cols   = 4
    rows   = math.ceil(len(imgs) / cols)
    colors = _class_colors(len(names))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4), squeeze=False)
    axes_flat = axes.flatten()
    for ax, img_path in zip(axes_flat, imgs):
        lbl_path = split_dir / 'train' / 'labels' / (img_path.stem + '.txt')
        bboxes, cls_ids = read_yolo_labels(lbl_path)
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        if bboxes:
            aug     = transform(image=img, bboxes=bboxes, class_labels=cls_ids)
            img     = aug['image']
            bboxes  = list(aug['bboxes'])
            cls_ids = [int(c) for c in aug['class_labels']]  # Albumentations puede devolver floats
        h, w = img.shape[:2]
        for (cx, cy, bw, bh), cls in zip(bboxes, cls_ids):
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            color = colors[cls % len(colors)]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = names[cls] if cls < len(names) else str(cls)
            cv2.putText(img, label, (x1, max(y1 - 4, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        ax.imshow(img)
        ax.axis('off')
    for ax in axes_flat[len(imgs):]:
        ax.axis('off')
    plt.suptitle('Preview augmentaciones - train split', fontsize=13)
    plt.tight_layout()
    plt.show()

# === Plots post-entrenamiento ===============================================

def show_training_plots(run_dir):
    for fname in ['results.png', 'confusion_matrix.png']:
        p = Path(run_dir) / fname
        if p.exists():
            print('--- {} ---'.format(fname))
            display(IPImage(str(p), width=900))

# === Metricas por clase =====================================================

def print_per_class_metrics(metrics, names):
    maps = list(metrics.box.maps)
    print()
    print('  {:<25}  {:>8}'.format('Clase', 'AP50-95'))
    print('  ' + '-' * 36)
    for i, ap in enumerate(maps):
        name = names[i] if i < len(names) else str(i)
        flag = '  !' if ap < 0.50 else ''
        print('  {:<25}  {:>8.4f}{}'.format(name, ap, flag))
    print()
    print('  mAP50    : {:.4f}'.format(metrics.box.map50))
    print('  mAP50-95 : {:.4f}'.format(metrics.box.map))

# === Guardar corrida en Drive ===============================================

def save_run_to_drive(run_dir, drive_output_dir, model_name, metrics=None):
    run_dir = Path(run_dir)
    out_dir = Path(drive_output_dir) / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['best.pt']:
        src = run_dir / 'weights' / fname
        if src.exists():
            shutil.copy2(src, out_dir / fname)
            print('  Guardado: {}'.format(out_dir / fname))
    for onnx_file in run_dir.rglob('*.onnx'):
        shutil.copy2(onnx_file, out_dir / 'best.onnx')
        print('  Guardado: {}'.format(out_dir / 'best.onnx'))
        break
    meta = {'model': model_name, 'date': datetime.now().isoformat()}
    if metrics is not None:
        meta['mAP50']    = float(metrics.box.map50)
        meta['mAP50_95'] = float(metrics.box.map)
    with open(out_dir / 'metrics.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print('  Guardado: metrics.json -> {}'.format(out_dir))

print('Utilidades cargadas')

## 4) Split de dataset

In [ ]:
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

names_yolo11 = load_names_from_yaml(DATASET_YOLO11 + '/data.yaml')
names_yolo26 = load_names_from_yaml(DATASET_YOLO26 + '/data.yaml')

print('Clases YOLO11 ({}): {}'.format(len(names_yolo11), names_yolo11))
print('Clases YOLO26 ({}): {}'.format(len(names_yolo26), names_yolo26))

stats_11 = split_single_dataset(DATASET_YOLO11, YOLO11_SPLIT_DIR, VAL_RATIO, TEST_RATIO, SEED)
stats_26 = split_single_dataset(DATASET_YOLO26, YOLO26_SPLIT_DIR, VAL_RATIO, TEST_RATIO, SEED)

print('Split YOLO11:', stats_11)
print('Split YOLO26:', stats_26)

## 5) Distribución de clases

Revisa que todas las clases tengan suficientes instancias.
Las barras en **rojo** indican clases con <50 instancias (riesgo de underfitting en esa clase).

In [ ]:
print('=== Distribucion de clases - YOLO11 (Cards) ===')
show_class_distribution(YOLO11_SPLIT_DIR, names_yolo11, 'YOLO11')

print()
print('=== Distribucion de clases - YOLO26 (Cards-2) ===')
show_class_distribution(YOLO26_SPLIT_DIR, names_yolo26, 'YOLO26')

## 6) Preview de augmentaciones

Define el transform y muestra cómo quedarán las imágenes **antes** de aplicarlo al dataset completo.
Verifica que los bounding boxes (coloreados por clase) sigan siendo correctos.

In [ ]:
# Define el transform — se reutiliza en apply_offline_augment
augment_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.35),
    A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=10, val_shift_limit=15, p=0.25),
    A.GaussNoise(p=0.20),
    A.MotionBlur(blur_limit=3, p=0.15),
    A.Affine(scale=(0.90, 1.10), translate_percent=(0.0, 0.04),
             rotate=(-8, 8), shear=(-3, 3), p=0.40),
    A.RandomShadow(p=0.15),
    A.CLAHE(p=0.15),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(1, 32), hole_width_range=(1, 32), p=0.20),
    A.ImageCompression(quality_range=(60, 100), p=0.15),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))

print('=== Preview augmentaciones - YOLO11 ===')
visualize_augmentations(YOLO11_SPLIT_DIR, augment_transform, names_yolo11, n=16)

print()
print('=== Preview augmentaciones - YOLO26 ===')
visualize_augmentations(YOLO26_SPLIT_DIR, augment_transform, names_yolo26, n=16)

## 7) Aplicar augmentación offline

Aplica el transform al split de `train` de cada dataset. Los splits `val` y `test` se copian sin cambios.

In [ ]:
if USE_OFFLINE_ALBUMENTATIONS:
    print('Aplicando Albumentations a YOLO11...')
    created_11 = apply_offline_augment(YOLO11_SPLIT_DIR, YOLO11_AUG_DIR, AUG_MULTIPLIER, augment_transform)
    print('Aplicando Albumentations a YOLO26...')
    created_26 = apply_offline_augment(YOLO26_SPLIT_DIR, YOLO26_AUG_DIR, AUG_MULTIPLIER, augment_transform)
    DATA_ROOT_YOLO11 = YOLO11_AUG_DIR
    DATA_ROOT_YOLO26 = YOLO26_AUG_DIR
    print('YOLO11 imagenes nuevas: {}'.format(created_11))
    print('YOLO26 imagenes nuevas: {}'.format(created_26))
else:
    DATA_ROOT_YOLO11 = YOLO11_SPLIT_DIR
    DATA_ROOT_YOLO26 = YOLO26_SPLIT_DIR
    print('Albumentations offline desactivado.')

for name, root in [('YOLO11', DATA_ROOT_YOLO11), ('YOLO26', DATA_ROOT_YOLO26)]:
    counts = {s: len(list((root / s / 'images').glob('*'))) for s in ['train', 'val', 'test']}
    print('{}  train:{} | val:{} | test:{}'.format(name, counts['train'], counts['val'], counts['test']))

## 8) Crear data.yaml

In [ ]:
YOLO11_DATA_YAML = BASE_DIR / 'cards_yolo11.yaml'
YOLO26_DATA_YAML = BASE_DIR / 'cards_yolo26.yaml'

build_yaml(DATA_ROOT_YOLO11, names_yolo11, YOLO11_DATA_YAML)
build_yaml(DATA_ROOT_YOLO26, names_yolo26, YOLO26_DATA_YAML)

print('YAML YOLO11:'); print(open(YOLO11_DATA_YAML).read())
print('YAML YOLO26:'); print(open(YOLO26_DATA_YAML).read())

## 9) Sección A — Entrenar YOLO11n

In [ ]:
model_11n = YOLO('yolo11n.pt')
results_11n = model_11n.train(
    data=str(YOLO11_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo11n_cards',
    plots=True,
    device=DEVICE,
    cache=True,
    amp=True,
    close_mosaic=10,
    cos_lr=COSINE_LR,
    nbs=NBS,
    mixup=MIXUP,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
)

## 10) Evaluar y exportar YOLO11n

In [ ]:
# Cargar mejor checkpoint
BEST_11N = RUNS_DIR / 'yolo11n_cards' / 'weights' / 'best.pt'
model_11n_best = YOLO(str(BEST_11N))

# Evaluar en test split
metrics_11n = model_11n_best.val(data=str(YOLO11_DATA_YAML), split='test')
print_per_class_metrics(metrics_11n, names_yolo11)

# Plots de entrenamiento
show_training_plots(RUNS_DIR / 'yolo11n_cards')

# Exportar a ONNX (se guarda junto a best.pt)
model_11n_best.export(format='onnx', dynamic=True, simplify=True)

# Guardar en Drive
save_run_to_drive(RUNS_DIR / 'yolo11n_cards', DRIVE_OUTPUT_DIR, 'yolo11n_cards', metrics_11n)

## 11) Sección B — Entrenar YOLO11s

In [ ]:
model_11s = YOLO('yolo11s.pt')
results_11s = model_11s.train(
    data=str(YOLO11_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo11s_cards',
    plots=True,
    device=DEVICE,
    cache=True,
    amp=True,
    close_mosaic=10,
    cos_lr=COSINE_LR,
    nbs=NBS,
    mixup=MIXUP,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
)

## 12) Evaluar y exportar YOLO11s

In [ ]:
BEST_11S = RUNS_DIR / 'yolo11s_cards' / 'weights' / 'best.pt'
model_11s_best = YOLO(str(BEST_11S))

metrics_11s = model_11s_best.val(data=str(YOLO11_DATA_YAML), split='test')
print_per_class_metrics(metrics_11s, names_yolo11)

show_training_plots(RUNS_DIR / 'yolo11s_cards')

model_11s_best.export(format='onnx', dynamic=True, simplify=True)

save_run_to_drive(RUNS_DIR / 'yolo11s_cards', DRIVE_OUTPUT_DIR, 'yolo11s_cards', metrics_11s)

## 13) Sección C — Entrenar YOLO26n

In [ ]:
model_26n = YOLO('yolo26n.pt')
results_26n = model_26n.train(
    data=str(YOLO26_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo26n_cards',
    plots=True,
    device=DEVICE,
    cache=True,
    amp=True,
    close_mosaic=10,
    cos_lr=COSINE_LR,
    nbs=NBS,
    mixup=MIXUP,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
)

## 14) Evaluar y exportar YOLO26n

In [ ]:
BEST_26 = RUNS_DIR / 'yolo26n_cards' / 'weights' / 'best.pt'
model_26_best = YOLO(str(BEST_26))

metrics_26n = model_26_best.val(data=str(YOLO26_DATA_YAML), split='test')
print_per_class_metrics(metrics_26n, names_yolo26)

show_training_plots(RUNS_DIR / 'yolo26n_cards')

model_26_best.export(format='onnx', dynamic=True, simplify=True)

save_run_to_drive(RUNS_DIR / 'yolo26n_cards', DRIVE_OUTPUT_DIR, 'yolo26n_cards', metrics_26n)

## 15) Sección D — Entrenar YOLO26s

In [ ]:
model_26s = YOLO('yolo26s.pt')
results_26s = model_26s.train(
    data=str(YOLO26_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name='yolo26s_cards',
    plots=True,
    device=DEVICE,
    cache=True,
    amp=True,
    close_mosaic=10,
    cos_lr=COSINE_LR,
    nbs=NBS,
    mixup=MIXUP,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
)

## 16) Evaluar y exportar YOLO26s

In [ ]:
BEST_26S = RUNS_DIR / 'yolo26s_cards' / 'weights' / 'best.pt'
model_26s_best = YOLO(str(BEST_26S))

metrics_26s = model_26s_best.val(data=str(YOLO26_DATA_YAML), split='test')
print_per_class_metrics(metrics_26s, names_yolo26)

show_training_plots(RUNS_DIR / 'yolo26s_cards')

model_26s_best.export(format='onnx', dynamic=True, simplify=True)

save_run_to_drive(RUNS_DIR / 'yolo26s_cards', DRIVE_OUTPUT_DIR, 'yolo26s_cards', metrics_26s)

## 17) Comparación de modelos

Ejecutar sólo después de haber corrido todas las secciones de entrenamiento.

In [ ]:
print('{:<14} {:>8} {:>10} {:>11}'.format('Modelo', 'mAP50', 'mAP50-95', 'Pesos .pt'))
print('-' * 47)

model_runs = [
    ('YOLO11n',  'metrics_11n',  'yolo11n_cards'),
    ('YOLO11s',  'metrics_11s',  'yolo11s_cards'),
    ('YOLO26n',  'metrics_26n',  'yolo26n_cards'),
    ('YOLO26s',  'metrics_26s',  'yolo26s_cards'),
]

g = globals()
for mname, var, run_name in model_runs:
    if var not in g:
        print('{:<14}  (seccion no ejecutada)'.format(mname))
        continue
    m  = g[var]
    pt = RUNS_DIR / run_name / 'weights' / 'best.pt'
    mb = os.path.getsize(str(pt)) / 1e6 if pt.exists() else float('nan')
    print('{:<14} {:>8.4f} {:>10.4f} {:>9.1f} MB'.format(
        mname, m.box.map50, m.box.map, mb
    ))

## Recomendaciones

- **Overfitting**: sube `AUG_MULTIPLIER` a `2`; verifica clases con <50 instancias.
- **mAP no mejora**: prueba `IMG_SIZE=768` con `BATCH` menor; revisa el learning rate.
- **Comparar**: usa `mAP50-95` como métrica principal + velocidad con `model.benchmark()`.
- **val/test**: nunca augmentar — son las métricas reales del modelo.
- **Drive**: `best.pt`, `last.pt`, `best.onnx` y `metrics.json` se guardan en
  `Cards_Training_Output/<modelo>/` automáticamente al ejecutar las celdas de eval/export.